In [1]:
from enum import Enum


class Library(str, Enum):
    DEEPEVAL = 'deepeval'
    RAGAS = 'ragas'
    TRULENS = 'trulens'
    PHOENIX = 'phoenix'
    MLFLOW = 'mlflow'
    RAGCHECKER = 'ragchecker'
    LLAMA_INDEX = 'llama_index'
    TONIC_VALIDATE = 'tonic_validate'

In [2]:
import os

from eval_fusion_core.utils.loaders import load_evaluation_inputs
# from eval_fusion_test.settings import get_openai_settings

In [3]:
from eval_fusion_deepeval.evaluator import DeepEvalEvaluator
from eval_fusion_deepeval.metrics import DeepEvalMetric
from eval_fusion_llama_index.evaluator import LlamaIndexEvaluator
from eval_fusion_llama_index.metrics import LlamaIndexMetric
from eval_fusion_mlflow.evaluator import MlFlowEvaluator
from eval_fusion_mlflow.metrics import MlFlowMetric
from eval_fusion_phoenix.evaluator import PhoenixEvaluator
from eval_fusion_phoenix.metrics import PhoenixMetric
from eval_fusion_ragas.evaluator import RagasEvaluator
from eval_fusion_ragas.metrics import RagasMetric
from eval_fusion_ragchecker.evaluator import RagCheckerEvaluator
from eval_fusion_ragchecker.metrics import RagCheckerMetric
from eval_fusion_tonic_validate.evaluator import TonicValidateEvaluator
from eval_fusion_tonic_validate.metrics import TonicValidateMetric
from eval_fusion_trulens.evaluator import TruLensEvaluator
from eval_fusion_trulens.metrics import TruLensMetric

W0728 23:29:04.230000 84171 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/munch/__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
from decouple import config
from eval_fusion_core.models.settings import EvalFusionEMSettings, EvalFusionLLMSettings
from eval_fusion_openai import OpenAIEM, OpenAILLM


def get_openai_settings(llm_name: str, em_name: str):
    llm_settings = EvalFusionLLMSettings(
        base_type=OpenAILLM,
        kwargs={
            'model_name': llm_name,
            'base_url': config('OPENAI_BASE_URL'),
            'api_key': config('OPENAI_API_KEY'),
        },
    )
    em_settings = EvalFusionEMSettings(
        base_type=OpenAIEM,
        kwargs={
            'model_name': em_name,
            'base_url': config('OPENAI_BASE_URL'),
            'api_key': config('OPENAI_API_KEY'),
        },
    )

    return llm_settings, em_settings

In [5]:
import inspect

from eval_fusion_core.base import EvalFusionBaseEvaluator, EvalFusionBaseMetric
from eval_fusion_core.models import EvaluationOutput, TokenUsage
from pydantic import BaseModel


class EvaluationResult(BaseModel):
    outputs: list[EvaluationOutput]
    token_usage: TokenUsage | tuple[TokenUsage, TokenUsage]


async def a_test_evaluator(
    evaluator_cls: type[EvalFusionBaseEvaluator],
    metrics: list[EvalFusionBaseMetric],
    llm_name: str,
    em_name: str,
) -> EvaluationResult:
    llm_settings, em_settings = get_openai_settings(llm_name, em_name)
    inputs = load_evaluation_inputs('../assets/amnesty_qa.json')

    signature = inspect.signature(evaluator_cls)
    has_em_settings = 'em_settings' in signature.parameters
    evaluator = (
        evaluator_cls(llm_settings, em_settings)
        if has_em_settings
        else evaluator_cls(llm_settings)
    )

    with evaluator:
        outputs = await evaluator.a_evaluate(
            inputs, metrics=metrics, feature=None, include_reason=False
        )

    return EvaluationResult(outputs=outputs, token_usage=evaluator.token_usage)

In [6]:
async def a_evaluate_by(
    llm_name: str,
    em_name: str,
    metric: EvalFusionBaseMetric,
    library: str,
    evaluator_cls: EvalFusionBaseEvaluator,
):
    result = await a_test_evaluator(evaluator_cls, [metric], llm_name, em_name)
    dir_path = f'outputs/{llm_name}/{library}'
    os.makedirs(dir_path, exist_ok=True)

    with open(f'{dir_path}/{metric.value}.json', 'w') as file:
        data = result.model_dump_json(indent=4)
        file.write(data)

## gpt-4o-mini

In [7]:
llm_name = 'gpt-4o-mini'
em_name = 'text-embedding-3-large'

# deepeval, ragas, mlflow, ragchecker, llama_index

### faithfulness

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

In [ ]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=RagasEvaluator,
)

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

In [ ]:
library = 'ragchecker'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagCheckerMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=RagCheckerEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### context_precision/contextual_precision

In [ ]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.CONTEXT_PRECISION,
    library=library,
    evaluator_cls=RagasEvaluator,
)

In [ ]:
library = 'ragchecker'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagCheckerMetric.CONTEXT_PRECISION,
    library=library,
    evaluator_cls=RagCheckerEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.CONTEXTUAL_PRECISION,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

### context_recall/contextual_recall

In [ ]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.CONTEXT_RECALL,
    library=library,
    evaluator_cls=RagasEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.CONTEXTUAL_RECALL,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

### answer_relevance/answer_relevancy/response_relevancy

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.ANSWER_RELEVANCE,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.ANSWER_RELEVANCY,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.ANSWER_RELEVANCY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

In [ ]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.RESPONSE_RELEVANCY,
    library=library,
    evaluator_cls=RagasEvaluator,
)

### answer_correctness/correctness

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.ANSWER_CORRECTNESS,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.CORRECTNESS,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### relevance/relevancy

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.RELEVANCE,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.RELEVANCY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### answer_similarity/semantic_similarity

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.ANSWER_SIMILARITY,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

In [ ]:
library = 'tonic_validate'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=TonicValidateMetric.ANSWER_SIMILARITY,
    library=library,
    evaluator_cls=TonicValidateEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.SEMANTIC_SIMILARITY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### context_relevance/context_relevancy/contextual_relevancy

In [ ]:
library = 'trulens'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=TruLensMetric.CONTEXT_RELEVANCE,
    library=library,
    evaluator_cls=TruLensEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.CONTEXT_RELEVANCY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.CONTEXTUAL_RELEVANCY,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)